# 02 · Data Cleaning & Feature Engineering

**Goal:** turn the raw bookings into a clean table and add useful features.

**Input:** `data/hotel_bookings.csv` · **Output:** `data/hotel_bookings_clean.csv`

Steps: inspect problems -> fix them -> engineer features.

In [1]:
import pandas as pd

df = pd.read_csv("../data/hotel_bookings.csv")

## 1. Inspect data-quality problems

In [2]:
print("Undefined market_segment:", (df["market_segment"] == "Undefined").sum())
print("Undefined distribution_channel:", (df["distribution_channel"] == "Undefined").sum())
print("Zero-night stays:", ((df["stays_in_weekend_nights"] + df["stays_in_week_nights"]) == 0).sum())

no_guests = (df["adults"] + df["children"] + df["babies"]) == 0
print("Zero guests:", no_guests.sum())
print("Negative adr:", (df["adr"] < 0).sum())
print("adr > 1000:", (df["adr"] > 1000).sum())
print("Babies > 3:", (df["babies"] > 3).sum())

Undefined market_segment: 2
Undefined distribution_channel: 5
Zero-night stays: 715
Zero guests: 180
Negative adr: 1
adr > 1000: 1
Babies > 3: 2


### Data leakage check

`reservation_status` is only known *after* the outcome (Canceled / Check-Out / No-Show), so it would leak
the answer. It is dropped together with `reservation_status_date`.

In [3]:
pd.crosstab(df["reservation_status"], df["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


## 2. Clean

In [4]:
df_clean = df.drop(columns=["reservation_status", "reservation_status_date"])

# Missing values
df_clean["children"] = df_clean["children"].fillna(0)
df_clean["country"] = df_clean["country"].fillna("Unknown")
df_clean["agent"] = df_clean["agent"].fillna(0)      # 0 = no agent
df_clean = df_clean.drop(columns=["company"])        # >90% missing

# 'Undefined' placeholders
df_clean["meal"] = df_clean["meal"].replace("Undefined", "SC")   # SC = no meal package
df_clean = df_clean[df_clean["market_segment"] != "Undefined"]
df_clean = df_clean[df_clean["distribution_channel"] != "Undefined"]

# Impossible / extreme values
no_guests = (df_clean["adults"] + df_clean["children"] + df_clean["babies"]) == 0
df_clean = df_clean[~no_guests]
df_clean = df_clean[(df_clean["adr"] >= 0) & (df_clean["adr"] < 1000)]
df_clean = df_clean[df_clean["babies"] <= 3]

print(df.shape, "->", df_clean.shape)
print("Nulls left:", df_clean.isnull().sum().sum())

(119390, 32) -> (119201, 29)
Nulls left: 0


## 3. Save the cleaned data

In [5]:
df_clean.to_csv("../data/hotel_bookings_clean.csv", index=False)

## 4. Feature engineering

| New feature | Meaning |
|---|---|
| `total_nights` | weekend + week nights |
| `total_guests` | adults + children + babies |
| `has_kids` | any children or babies |
| `has_agent` | booked through an agent |
| `ever_canceled_before` | guest has cancelled previously |
| `arrival_month` | month as a number (1-12) |
| `country` | top 10 kept, rest grouped as `Other` |

In [6]:
df_fe = df_clean.copy()

df_fe["total_nights"] = df_fe["stays_in_weekend_nights"] + df_fe["stays_in_week_nights"]
df_fe["total_guests"] = df_fe["adults"] + df_fe["children"] + df_fe["babies"]
df_fe["has_kids"] = ((df_fe["children"] + df_fe["babies"]) > 0).astype(int)
df_fe["has_agent"] = (df_fe["agent"] != 0).astype(int)
df_fe["ever_canceled_before"] = (df_fe["previous_cancellations"] > 0).astype(int)

months = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
df_fe["arrival_month"] = df_fe["arrival_date_month"].map({m: i + 1 for i, m in enumerate(months)})

top_countries = df_fe["country"].value_counts().head(10).index
df_fe["country"] = df_fe["country"].where(df_fe["country"].isin(top_countries), "Other")

### Drop columns that do not help

- `arrival_date_month` / `arrival_date_year` -> replaced by `arrival_month` (year would not generalise to future bookings)
- `agent` -> replaced by `has_agent`
- `assigned_room_type` -> assigned *after* booking, so it can leak the outcome

In [7]:
df_fe = df_fe.drop(columns=["arrival_date_month", "arrival_date_year", "agent", "assigned_room_type"])
print(df_fe.shape)

(119201, 31)


## 5. Sanity check: do the new features relate to the target?

In [8]:
print(df_fe.groupby("ever_canceled_before")["is_canceled"].mean().round(3), "\n")
print(df_fe.groupby("has_kids")["is_canceled"].mean().round(3))

ever_canceled_before
0    0.339
1    0.917
Name: is_canceled, dtype: float64 

has_kids
0    0.373
1    0.349
Name: is_canceled, dtype: float64
